# All Run Submissions in One Plot

Plots every `runs/*/submission.csv` target series on a single figure.

In [4]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.')
RUNS_DIR = ROOT / 'runs'

submission_paths = sorted(RUNS_DIR.glob('*/submission.csv'))
extra_paths = [p for p in [ROOT / 'submissio.csv', ROOT / 'submission.csv', ROOT / 'submission_per_market.csv'] if p.exists()]

all_paths = submission_paths + extra_paths
print(f'Found {len(submission_paths)} run submissions and {len(extra_paths)} root submissions.')
for p in all_paths[:20]:
    print('-', p)
if len(all_paths) > 20:
    print(f'... and {len(all_paths)-10} more')

Found 14 run submissions and 3 root submissions.
- runs/20260219-1146_98a759c_per_market_v1/submission.csv
- runs/20260219-1345_98a759c_per_market_v1/submission.csv
- runs/20260219-1642_5599aaf_per_market_v1/submission.csv
- runs/20260219-1705_92239c7_per_market_v1/submission.csv
- runs/20260219-1741_c3b064f_per_market_v2/submission.csv
- runs/20260219-1809_c3b064f_per_market_v2/submission.csv
- runs/20260219-1826_c33f25d_per_market_v2_noglobaldelta/submission.csv
- runs/20260219-1835_c33f25d_per_market_v2_with_globaldelta/submission.csv
- runs/20260219-1848_fd873d2_per_market_v2_peakexpert/submission.csv
- runs/20260219-1904_6e61965_per_market_v2_peak_horizon/submission.csv
- runs/20260219-1916_6e61965_per_market_v2_exoglags/submission.csv
- runs/20260220-101135_intraday_submit_v1/submission.csv
- runs/20260220-105456_intraday_cv_smoke/submission.csv
- runs/20260220-105809_intraday_cv_fast/submission.csv
- submissio.csv
- submission.csv
- submission_per_market.csv


In [5]:
series = []
for path in all_paths:
    try:
        df = pd.read_csv(path, usecols=['id', 'target'])
    except Exception as e:
        print(f'Skipping {path}: {e}')
        continue
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df['target'] = pd.to_numeric(df['target'], errors='coerce')
    df = df.dropna().sort_values('id')
    if df.empty:
        continue
    label = path.parent.name if path.parent.name != '.' else path.name
    if path.parent.name == '.':
        label = f'root:{path.name}'
    series.append((label, df['id'].to_numpy(), df['target'].to_numpy()))

print(f'Ready to plot {len(series)} series.')

Ready to plot 17 series.


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
fig = go.Figure()

for label, x, y in series:
    line_width = 1
    opacity = 0.22
    if label.startswith('root:submissio.csv'):
        line_width = 2
        opacity = 0.95
    fig.add_trace(
        go.Scatter(
            x=x, y=y, mode='lines', name=label,
            line=dict(width=line_width), opacity=opacity,
            hovertemplate='id=%{x}<br>target=%{y:.4f}<extra>'+label+'</extra>'
        )
    )

fig.update_layout(
    title='All Submission Targets (Interactive)',
    xaxis_title='id',
    yaxis_title='target',
    template='plotly_white',
    height=620,
    hovermode='x unified',
)
fig.show()


In [ ]:
import math

n = len(series)
if n == 0:
    print('No series to plot.')
else:
    cols = 3
    rows = math.ceil(n / cols)
    subplot_titles = [label for label, _, _ in series]
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=subplot_titles, vertical_spacing=0.06)

    for i, (label, x, y) in enumerate(series, start=1):
        r = (i - 1) // cols + 1
        c = (i - 1) % cols + 1
        fig.add_trace(
            go.Scatter(
                x=x, y=y, mode='lines', name=label, showlegend=False,
                line=dict(width=1.2),
                hovertemplate='id=%{x}<br>target=%{y:.4f}<extra>'+label+'</extra>'
            ),
            row=r, col=c
        )

    fig.update_layout(
        title='Individual Submission Target Plots (Interactive)',
        template='plotly_white',
        height=max(500, 280 * rows),
    )
    fig.update_xaxes(title_text='id')
    fig.update_yaxes(title_text='target')
    fig.show()


In [ ]:
from pathlib import Path
import pandas as pd

p = Path('submissio.csv')
if not p.exists():
    print('submissio.csv not found')
else:
    df = pd.read_csv(p, usecols=['id', 'target'])
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df['target'] = pd.to_numeric(df['target'], errors='coerce')
    df = df.dropna().sort_values('id')

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df['id'], y=df['target'], mode='lines', name='submissio.csv',
            line=dict(width=1.6),
            hovertemplate='id=%{x}<br>target=%{y:.4f}<extra>submissio.csv</extra>'
        )
    )
    fig.update_layout(
        title='submissio.csv target (Interactive)',
        xaxis_title='id',
        yaxis_title='target',
        template='plotly_white',
        height=520,
    )
    fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go

test_path = 'data/test_for_participants.csv'
test = pd.read_csv(test_path)

core_non_meteo = {'id','market','delivery_start','delivery_end','load_forecast','wind_forecast','solar_forecast'}
meteo_cols = [c for c in test.columns if c not in core_non_meteo]

test['id'] = pd.to_numeric(test['id'], errors='coerce')
test = test.dropna(subset=['id']).sort_values('id')
test['meteo_missing_count'] = test[meteo_cols].isna().sum(axis=1)
test['meteo_missing_any'] = (test['meteo_missing_count'] > 0).astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=test['id'], y=test['meteo_missing_count'], mode='lines',
    name='test missing count', line=dict(width=1.6), opacity=0.95,
    hovertemplate='id=%{x}<br>missing_count=%{y}<extra>test</extra>'
))
fig.add_trace(go.Scatter(
    x=test.loc[test['meteo_missing_any'] == 1, 'id'],
    y=test.loc[test['meteo_missing_any'] == 1, 'meteo_missing_count'],
    mode='markers', name='rows with any meteo missing',
    marker=dict(size=4), opacity=0.85,
    hovertemplate='id=%{x}<br>missing_count=%{y}<extra>test missing</extra>'
))

fig.update_layout(
    title='Test Weather Missingness over IDs',
    xaxis_title='id',
    yaxis_title='meteo missing count',
    template='plotly_white',
    height=520,
    hovermode='x unified'
)
fig.show()

print('Test missing-any rate:', f"{test['meteo_missing_any'].mean():.4%}")
print('Num meteo columns:', len(meteo_cols))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

base_path = Path("submissio.csv")
imputed_candidates = sorted(
    Path("runs").glob("**/submission_baseline_imputed_simple_on_missing.csv"),
    key=lambda p: p.stat().st_mtime,
)
if not imputed_candidates:
    raise FileNotFoundError("No imputed submission found under runs/**/submission_baseline_imputed_simple_on_missing.csv")

imputed_path = imputed_candidates[-1]
print(f"Using imputed file: {imputed_path}")

base = pd.read_csv(base_path)
imp = pd.read_csv(imputed_path)

for df, name in [(base, "submissio.csv"), (imp, str(imputed_path))]:
    if not {"id", "target"}.issubset(df.columns):
        raise ValueError(f"{name} must contain id,target columns")

cmp = (
    base[["id", "target"]]
    .rename(columns={"target": "target_base"})
    .merge(
        imp[["id", "target"]].rename(columns={"target": "target_imputed"}),
        on="id",
        how="inner",
    )
    .sort_values("id")
)
cmp["diff"] = cmp["target_imputed"] - cmp["target_base"]
cmp["abs_diff"] = cmp["diff"].abs()
cmp["changed"] = cmp["abs_diff"] > 1e-12

changed = cmp[cmp["changed"]]
print(f"Overlap rows: {len(cmp)}")
print(f"Changed rows: {len(changed)} ({len(changed)/len(cmp):.2%})")
if len(changed):
    print(f"Mean abs diff on changed rows: {changed['abs_diff'].mean():.6f}")
    print(f"P95 abs diff on changed rows: {np.percentile(changed['abs_diff'],95):.6f}")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "submissio.csv vs imputed submission",
        "Signed and absolute difference (imputed - baseline)",
    ),
)

fig.add_trace(
    go.Scatter(
        x=cmp["id"],
        y=cmp["target_base"],
        mode="lines",
        name="submissio.csv",
        line=dict(color="#1f77b4", width=1.6),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=cmp["id"],
        y=cmp["target_imputed"],
        mode="lines",
        name="imputed",
        line=dict(color="#ff7f0e", width=1.3),
    ),
    row=1,
    col=1,
)

if len(changed):
    fig.add_trace(
        go.Scatter(
            x=changed["id"],
            y=changed["target_imputed"],
            mode="markers",
            name="changed IDs",
            marker=dict(color="#d62728", size=5, opacity=0.85),
            hovertemplate="id=%{x}<br>imputed=%{y:.3f}<br>diff=%{customdata:.3f}<extra></extra>",
            customdata=changed["diff"],
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=cmp["id"],
        y=cmp["diff"],
        mode="lines",
        name="signed diff",
        line=dict(color="#9467bd", width=1.2),
    ),
    row=2,
    col=1,
)
fig.add_hline(y=0.0, line_width=1, line_dash="dash", line_color="#666", row=2, col=1)

fig.add_trace(
    go.Scatter(
        x=cmp["id"],
        y=cmp["abs_diff"],
        mode="lines",
        name="abs diff",
        line=dict(color="#2ca02c", width=1.6),
    ),
    row=2,
    col=1,
)
if len(changed):
    fig.add_trace(
        go.Scatter(
            x=changed["id"],
            y=changed["abs_diff"],
            mode="markers",
            name="changed abs diff",
            marker=dict(color="#d62728", size=4, opacity=0.85),
        ),
        row=2,
        col=1,
    )

fig.update_layout(
    title="submissio.csv vs baseline-imputed submission (Interactive)",
    template="plotly_white",
    height=800,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig.update_xaxes(title_text="id", row=2, col=1)
fig.update_yaxes(title_text="target", row=1, col=1)
fig.update_yaxes(title_text="diff", row=2, col=1)
fig.show()


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

base_path = Path("submissio.csv")
test_path = Path("data/test_for_participants.csv")
imputed_candidates = sorted(
    Path("runs").glob("**/submission_baseline_imputed_simple_on_missing.csv"),
    key=lambda p: p.stat().st_mtime,
)
if not imputed_candidates:
    raise FileNotFoundError("No imputed submission found under runs/**/submission_baseline_imputed_simple_on_missing.csv")
imputed_path = imputed_candidates[-1]

base = pd.read_csv(base_path)
imp = pd.read_csv(imputed_path)
test_df = pd.read_csv(test_path, usecols=["id", "market"]) 

cmp = (
    base[["id", "target"]]
    .rename(columns={"target": "target_base"})
    .merge(imp[["id", "target"]].rename(columns={"target": "target_imputed"}), on="id", how="inner")
    .merge(test_df, on="id", how="left")
    .sort_values(["market", "id"])
)
cmp["diff"] = cmp["target_imputed"] - cmp["target_base"]
cmp["abs_diff"] = cmp["diff"].abs()
cmp["changed"] = cmp["abs_diff"] > 1e-12

markets = [m for m in sorted(cmp["market"].dropna().unique())]
if not markets:
    raise ValueError("No market labels found after merging with test data")

print(f"Using imputed file: {imputed_path}")
print(f"Markets: {markets}")

# Figure 1: per-market target overlay
fig_targets = make_subplots(
    rows=len(markets),
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.03,
    subplot_titles=[f"{m}" for m in markets],
)

for i, m in enumerate(markets, start=1):
    d = cmp[cmp["market"] == m].sort_values("id")
    ch = d[d["changed"]]

    fig_targets.add_trace(
        go.Scatter(
            x=d["id"],
            y=d["target_base"],
            mode="lines",
            name="submissio.csv",
            legendgroup="base",
            showlegend=(i == 1),
            line=dict(color="#1f77b4", width=1.2),
        ),
        row=i,
        col=1,
    )
    fig_targets.add_trace(
        go.Scatter(
            x=d["id"],
            y=d["target_imputed"],
            mode="lines",
            name="imputed",
            legendgroup="imp",
            showlegend=(i == 1),
            line=dict(color="#ff7f0e", width=1.1),
        ),
        row=i,
        col=1,
    )
    if len(ch):
        fig_targets.add_trace(
            go.Scatter(
                x=ch["id"],
                y=ch["target_imputed"],
                mode="markers",
                name="changed IDs",
                legendgroup="changed",
                showlegend=(i == 1),
                marker=dict(color="#d62728", size=4, opacity=0.8),
                hovertemplate="market={m}<br>id=%{x}<br>imputed=%{y:.3f}<br>diff=%{customdata:.3f}<extra></extra>",
                customdata=ch["diff"],
            ),
            row=i,
            col=1,
        )

fig_targets.update_layout(
    title="Per-market: submissio.csv vs imputed",
    template="plotly_white",
    height=max(350 * len(markets), 900),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0.0),
)
for i in range(1, len(markets) + 1):
    fig_targets.update_xaxes(title_text="id", row=i, col=1)
    fig_targets.update_yaxes(title_text="target", row=i, col=1)
fig_targets.show()

# Figure 2: per-market diff panel (signed + abs)
fig_diff = make_subplots(
    rows=len(markets),
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.03,
    subplot_titles=[f"{m}" for m in markets],
)

for i, m in enumerate(markets, start=1):
    d = cmp[cmp["market"] == m].sort_values("id")
    ch = d[d["changed"]]

    fig_diff.add_trace(
        go.Scatter(
            x=d["id"],
            y=d["diff"],
            mode="lines",
            name="signed diff",
            legendgroup="sdiff",
            showlegend=(i == 1),
            line=dict(color="#9467bd", width=1.0),
        ),
        row=i,
        col=1,
    )
    fig_diff.add_trace(
        go.Scatter(
            x=d["id"],
            y=d["abs_diff"],
            mode="lines",
            name="abs diff",
            legendgroup="adiff",
            showlegend=(i == 1),
            line=dict(color="#2ca02c", width=1.1),
        ),
        row=i,
        col=1,
    )
    if len(ch):
        fig_diff.add_trace(
            go.Scatter(
                x=ch["id"],
                y=ch["abs_diff"],
                mode="markers",
                name="changed abs diff",
                legendgroup="chad",
                showlegend=(i == 1),
                marker=dict(color="#d62728", size=4, opacity=0.8),
            ),
            row=i,
            col=1,
        )
    fig_diff.add_hline(y=0.0, line_width=1, line_dash="dash", line_color="#666", row=i, col=1)

fig_diff.update_layout(
    title="Per-market differences (imputed - baseline)",
    template="plotly_white",
    height=max(350 * len(markets), 900),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0.0),
)
for i in range(1, len(markets) + 1):
    fig_diff.update_xaxes(title_text="id", row=i, col=1)
    fig_diff.update_yaxes(title_text="diff", row=i, col=1)
fig_diff.show()

summary = (
    cmp.groupby("market", dropna=False)
    .agg(
        rows=("id", "count"),
        changed_rows=("changed", "sum"),
        mean_signed_diff=("diff", "mean"),
        mean_abs_diff=("abs_diff", "mean"),
        p95_abs_diff=("abs_diff", lambda s: np.percentile(s, 95)),
    )
    .reset_index()
)
summary["changed_rate"] = summary["changed_rows"] / summary["rows"]
display(summary.sort_values("mean_abs_diff", ascending=False))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

base_path = Path("submissio.csv")
plus2_path = Path("submission_submissio_plus2std_missing.csv")

base = pd.read_csv(base_path)
new = pd.read_csv(plus2_path)

cmp = (
    base[["id", "target"]]
    .rename(columns={"target": "target_base"})
    .merge(new[["id", "target"]].rename(columns={"target": "target_plus2"}), on="id", how="inner")
    .sort_values("id")
)
cmp["diff"] = cmp["target_plus2"] - cmp["target_base"]
cmp["abs_diff"] = cmp["diff"].abs()
cmp["changed"] = cmp["abs_diff"] > 1e-12
changed = cmp[cmp["changed"]]

print(f"Rows: {len(cmp)}")
print(f"Changed rows: {len(changed)} ({len(changed)/len(cmp):.2%})")
if len(changed):
    print(f"Mean signed diff on changed rows: {changed['diff'].mean():.6f}")
    print(f"Mean abs diff on changed rows: {changed['abs_diff'].mean():.6f}")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "submissio.csv vs submission_submissio_plus2std_missing.csv",
        "Signed and absolute difference (plus2std - baseline)",
    ),
)

fig.add_trace(
    go.Scatter(x=cmp["id"], y=cmp["target_base"], mode="lines", name="submissio.csv", line=dict(color="#1f77b4", width=1.6)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=cmp["id"], y=cmp["target_plus2"], mode="lines", name="plus2std", line=dict(color="#ff7f0e", width=1.2)),
    row=1, col=1,
)
if len(changed):
    fig.add_trace(
        go.Scatter(
            x=changed["id"], y=changed["target_plus2"], mode="markers", name="changed IDs",
            marker=dict(color="#d62728", size=5, opacity=0.85), customdata=changed["diff"],
            hovertemplate="id=%{x}<br>plus2=%{y:.3f}<br>diff=%{customdata:.3f}<extra></extra>",
        ),
        row=1, col=1,
    )

fig.add_trace(
    go.Scatter(x=cmp["id"], y=cmp["diff"], mode="lines", name="signed diff", line=dict(color="#9467bd", width=1.2)),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=cmp["id"], y=cmp["abs_diff"], mode="lines", name="abs diff", line=dict(color="#2ca02c", width=1.3)),
    row=2, col=1,
)
if len(changed):
    fig.add_trace(
        go.Scatter(x=changed["id"], y=changed["abs_diff"], mode="markers", name="changed abs diff", marker=dict(color="#d62728", size=4, opacity=0.85)),
        row=2, col=1,
    )
fig.add_hline(y=0.0, line_width=1, line_dash="dash", line_color="#666", row=2, col=1)

fig.update_layout(
    title="submissio.csv vs plus2std-missing submission (Interactive)",
    template="plotly_white",
    height=800,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig.update_xaxes(title_text="id", row=2, col=1)
fig.update_yaxes(title_text="target", row=1, col=1)
fig.update_yaxes(title_text="diff", row=2, col=1)
fig.show()


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

base_path = Path("submissio.csv")
plus2_path = Path("submission_submissio_plus2std_missing.csv")
test_path = Path("data/test_for_participants.csv")

base = pd.read_csv(base_path)
new = pd.read_csv(plus2_path)
test_df = pd.read_csv(test_path, usecols=["id", "market"])

cmp = (
    base[["id", "target"]]
    .rename(columns={"target": "target_base"})
    .merge(new[["id", "target"]].rename(columns={"target": "target_plus2"}), on="id", how="inner")
    .merge(test_df, on="id", how="left")
    .sort_values(["market", "id"])
)
cmp["diff"] = cmp["target_plus2"] - cmp["target_base"]
cmp["abs_diff"] = cmp["diff"].abs()
cmp["changed"] = cmp["abs_diff"] > 1e-12

markets = sorted(cmp["market"].dropna().unique())
print("Markets:", markets)

fig_targets = make_subplots(
    rows=len(markets), cols=1, shared_xaxes=False, vertical_spacing=0.03,
    subplot_titles=[f"{m}" for m in markets],
)

for i, m in enumerate(markets, start=1):
    d = cmp[cmp["market"] == m].sort_values("id")
    ch = d[d["changed"]]

    fig_targets.add_trace(
        go.Scatter(x=d["id"], y=d["target_base"], mode="lines", name="submissio.csv", legendgroup="b", showlegend=(i==1), line=dict(color="#1f77b4", width=1.2)),
        row=i, col=1,
    )
    fig_targets.add_trace(
        go.Scatter(x=d["id"], y=d["target_plus2"], mode="lines", name="plus2std", legendgroup="p", showlegend=(i==1), line=dict(color="#ff7f0e", width=1.1)),
        row=i, col=1,
    )
    if len(ch):
        fig_targets.add_trace(
            go.Scatter(x=ch["id"], y=ch["target_plus2"], mode="markers", name="changed IDs", legendgroup="c", showlegend=(i==1), marker=dict(color="#d62728", size=4, opacity=0.85)),
            row=i, col=1,
        )

fig_targets.update_layout(
    title="Per-market: submissio.csv vs plus2std-missing",
    template="plotly_white",
    height=max(350 * len(markets), 900),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0.0),
)
for i in range(1, len(markets)+1):
    fig_targets.update_xaxes(title_text="id", row=i, col=1)
    fig_targets.update_yaxes(title_text="target", row=i, col=1)
fig_targets.show()

fig_diff = make_subplots(
    rows=len(markets), cols=1, shared_xaxes=False, vertical_spacing=0.03,
    subplot_titles=[f"{m}" for m in markets],
)

for i, m in enumerate(markets, start=1):
    d = cmp[cmp["market"] == m].sort_values("id")
    ch = d[d["changed"]]

    fig_diff.add_trace(
        go.Scatter(x=d["id"], y=d["diff"], mode="lines", name="signed diff", legendgroup="sd", showlegend=(i==1), line=dict(color="#9467bd", width=1.0)),
        row=i, col=1,
    )
    fig_diff.add_trace(
        go.Scatter(x=d["id"], y=d["abs_diff"], mode="lines", name="abs diff", legendgroup="ad", showlegend=(i==1), line=dict(color="#2ca02c", width=1.1)),
        row=i, col=1,
    )
    if len(ch):
        fig_diff.add_trace(
            go.Scatter(x=ch["id"], y=ch["abs_diff"], mode="markers", name="changed abs diff", legendgroup="cad", showlegend=(i==1), marker=dict(color="#d62728", size=4, opacity=0.85)),
            row=i, col=1,
        )
    fig_diff.add_hline(y=0.0, line_width=1, line_dash="dash", line_color="#666", row=i, col=1)

fig_diff.update_layout(
    title="Per-market differences (plus2std - baseline)",
    template="plotly_white",
    height=max(350 * len(markets), 900),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0.0),
)
for i in range(1, len(markets)+1):
    fig_diff.update_xaxes(title_text="id", row=i, col=1)
    fig_diff.update_yaxes(title_text="diff", row=i, col=1)
fig_diff.show()

summary = (
    cmp.groupby("market", dropna=False)
    .agg(
        rows=("id", "count"),
        changed_rows=("changed", "sum"),
        mean_signed_diff=("diff", "mean"),
        mean_abs_diff=("abs_diff", "mean"),
        p95_abs_diff=("abs_diff", lambda s: np.percentile(s, 95)),
    )
    .reset_index()
)
summary["changed_rate"] = summary["changed_rows"] / summary["rows"]
display(summary.sort_values("mean_abs_diff", ascending=False))
